## Feature Engineering

In [96]:
import pandas as pd

df = pd.read_csv("../data/telco_cleaned.csv")

### Target variable Ecoding

In [97]:
df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

df["Churn"].value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

In [98]:
# Many features have yes/no values, I am going to encode them before feeding data to the model.

for col in df.columns:
    print(f"Column: {col}{df[col].unique (  )}")

Column: gender<StringArray>
['Female', 'Male']
Length: 2, dtype: str
Column: SeniorCitizen[0 1]
Column: Partner<StringArray>
['Yes', 'No']
Length: 2, dtype: str
Column: Dependents<StringArray>
['No', 'Yes']
Length: 2, dtype: str
Column: tenure[ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26 39]
Column: PhoneService<StringArray>
['No', 'Yes']
Length: 2, dtype: str
Column: MultipleLines<StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str
Column: InternetService<StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str
Column: OnlineSecurity<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
Column: OnlineBackup<StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str
Column: DeviceProtection<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
Col

### Binary Encoding

In [99]:
binary_cols = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]

for col in binary_cols:
    df[col] = df[col].map({
        "Yes": 1,
        "No": 0
    })

for col in binary_cols:
    print(f"Column: {col}, Unique Values: {df[col].unique()}")

Column: Partner, Unique Values: [1 0]
Column: Dependents, Unique Values: [0 1]
Column: PhoneService, Unique Values: [0 1]
Column: PaperlessBilling, Unique Values: [1 0]


### Gender Encoding

In [100]:
df["gender"] = df["gender"].map({
    "Male" : 1,
    "Female" : 0
})

df["gender"].value_counts()

gender
1    3549
0    3483
Name: count, dtype: int64

### Tenure Grouping

In [101]:
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[0,12,24,48,72],
    labels=[
        "0-12",
        "12-24",
        "24-48",
        "48-72"
    ]
)

### Average Monthly Spent 

In [102]:
df["AvgMonthlySpend"] = (
    df["TotalCharges"] /
    (df["tenure"] + 1)
).round(2)

### Calculating High Value Customers

In [103]:
df["High_value_customer"] = (df["MonthlyCharges"] > df["MonthlyCharges"].median()).astype(int)

### Contract length feature

In [104]:
contract_map = {
    "Month-to-month":0,
    "One year":1,
    "Two year":2
}

df["ContractLength"] = df["Contract"].map(contract_map)

### Total Services Count

In [105]:
services = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["TotalServices"] = 0

for col in services:
    df["TotalServices"] += (
        df[col].isin(["Yes"])
    ).astype(int)

### Monthly Charge per Service

In [106]:
df["ChargePerService"] = (
    df["MonthlyCharges"] /
    (df["TotalServices"] + 1)
).round(2)

In [ ]:
df["IsNewCustomer"] = (df["tenure"] <= 12).astype(int)

df["HasSecurityServices"] = (
    (df["OnlineSecurity"] == "Yes") |
    (df["TechSupport"] == "Yes")
).astype(int)

### Saperating Features


In [107]:
cat_cols = df.select_dtypes(include = "object").columns

print("Categorical Columns: ", cat_cols)

Categorical Columns:  Index(['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaymentMethod'],
      dtype='str')


C:\Users\Angel\AppData\Local\Temp\ipykernel_5752\615412640.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include = "object").columns


In [108]:
num_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

### Train Test Split

In [109]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TenureGroup,AvgMonthlySpend,High_value_customer,ContractLength,TotalServices,ChargePerService
0,0,0,1,0,1,0,No phone service,DSL,No,Yes,...,Electronic check,29.85,29.85,0,0-12,14.92,0,0,1,14.92
1,1,0,0,0,34,1,No,DSL,Yes,No,...,Mailed check,56.95,1889.50,0,24-48,53.99,0,1,2,18.98
2,1,0,0,0,2,1,No,DSL,Yes,Yes,...,Mailed check,53.85,108.15,1,0-12,36.05,0,0,2,17.95
3,1,0,0,0,45,0,No phone service,DSL,Yes,No,...,Bank transfer (automatic),42.30,1840.75,0,24-48,40.02,0,1,3,10.58
4,0,0,0,0,2,1,No,Fiber optic,No,No,...,Electronic check,70.70,151.65,1,0-12,50.55,1,0,0,70.70


In [110]:
from sklearn.model_selection import train_test_split

X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Using Column Transformer

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder


In [114]:


preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(drop="first",
                              handle_unknown="ignore"),
         cat_cols)
    ]
)